<a href="https://colab.research.google.com/github/ShiYu0318/ShiYu-AI-Courses-Archive/blob/main/LLM_RAG_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai gradio pymupdf ragas

In [2]:
import os
import fitz
import numpy as np
import gradio as gr
from google.colab import userdata
from google.colab import files
from google import genai
from google.genai import types


os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
client = genai.Client()
print("Client ready:", client is not None)

Client ready: True


In [4]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="用一句話介紹中央大學"
)
print(response.text)

中央大學是一所位於桃園的頂尖研究型大學，尤其以地球科學、太空科學、光電與資訊工程等理工領域享有盛譽。


In [5]:
def chat_with_gemini(message, history):
    """把對話歷史送進 Gemini，回傳新訊息。"""
    # 把 Gradio 的 history 格式轉成 Gemini 接受的格式
    contents = []
    for msg in history:
        role = "user" if msg["role"] == "user" else "model"
        contents.append({"role": role, "parts": [{"text": msg["content"]}]})
    contents.append({"role": "user", "parts": [{"text": message}]})

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=contents
    )
    return response.text

# 啟動 chatbot
gr.ChatInterface(
    fn=chat_with_gemini,
    title="My First AI Chatbot",
    description="Power by Gemini 2.5 Flash",
    type="messages",
).launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a58816a15f6f21121.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
def extract_pdf(path: str) -> str:
    doc = fitz.open(path)
    return "\n".join(page.get_text() for page in doc)

def chunk_text(text: str, chunk_size: int = 100, overlap: int = 20) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += chunk_size - overlap
    return [c for c in chunks if c.strip()]

In [ ]:
def embed_docs(texts: list[str]) -> np.ndarray:
    vecs = []
    for i, t in enumerate(texts):
        r = client.models.embed_content(
            model="gemini-embedding-001",
            contents=t,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
        )
        vecs.append(r.embeddings[0].values)
        if (i + 1) % 10 == 0:
            print(f"  embedded {i+1}/{len(texts)}...")
    return np.array(vecs)

def embed_query(text: str) -> np.ndarray:
    r = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    return np.array(r.embeddings[0].values)

In [7]:
# 把檔名改成你上傳的 PDF 名稱
pdf_path = "/content/國立中央大學學生社團輔導辦法.pdf"
print(f"Upload Finish：{pdf_path}\n")

raw_text = extract_pdf(pdf_path)
chunks   = chunk_text(raw_text)
print(f"Extract Finish：{len(raw_text)} words, {len(chunks)} chunks\n")
print("Embedding...\n")

doc_embeddings = embed_docs(chunks)
print(f"\nEmbedding Finish：{doc_embeddings.shape}")

Upload Finish：/content/國立中央大學學生社團輔導辦法.pdf

Extract Finish：2756 words, 35 chunks

Embedding...

  embedded 10/35...
  embedded 20/35...
  embedded 30/35...

Embedding Finish：(35, 3072)


In [8]:
def retrieve(query: str, k: int = 3) -> list[str]:
    q_vec = embed_query(query)
    q_norm = q_vec / np.linalg.norm(q_vec)
    d_norm = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    sims   = d_norm @ q_norm
    top_k  = np.argsort(sims)[::-1][:k]
    return [chunks[i] for i in top_k]

def chat_with_rag(message: str, history: list) -> str:
    retrieved = retrieve(message, k=3)
    context   = "\n\n".join(f"[段落 {i+1}]\n{c}" for i, c in enumerate(retrieved))

    # 帶對話歷史的 contents
    contents = []
    for msg in history:
        role = "user" if msg["role"] == "user" else "model"
        contents.append({"role": role, "parts": [{"text": msg["content"]}]})
    contents.append({"role": "user", "parts": [{"text": message}]})

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=(
                f"你是一個 PDF 文件問答助手。請只根據以下資料回答問題；"
                f"若資料中找不到答案，請說「文件中沒有相關資訊」，不要編造。\n\n"
                f"【參考資料】\n{context}"
            )
        ),
    )
    return response.text

gr.ChatInterface(
    fn=chat_with_rag,
    title=f"AI Chatbot with RAG",
    description="根據上傳的 PDF 回答問題",
    type="messages",
    examples=[
        ["這份文件的主要內容是什麼？"],
        ["幫我摘要這份文件的重點"],
    ],
).launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7e93349c0453bdf2fb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
